# 🤪 WGAN - CelebA Faces

In this notebook, we'll walk through the steps required to train your own Wasserstein GAN on the CelebA faces dataset

The code has been adapted from the excellent [WGAN-GP tutorial](https://keras.io/examples/generative/wgan_gp/) created by Aakash Kumar Nain, available on the Keras website.

In [ ]:
%load_ext autoreload
%autoreload 2
# TensorBoard imports TensorFlow whenever it is installed, and TensorFlow and PyTorch
# in one process can crash. This makes TensorBoard use its own TF-free version instead
import sys, types
sys.modules["tensorboard.compat.notf"] = types.ModuleType("tensorboard.compat.notf")

import itertools
import numpy as np
from pathlib import Path
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display, sample_batch

# PyTorch needs to be told which device to use
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 64
CHANNELS = 3
BATCH_SIZE = 512
NUM_FEATURES = 64
Z_DIM = 128
LEARNING_RATE = 0.0002
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.999
EPOCHS = 200
CRITIC_STEPS = 3
GP_WEIGHT = 10.0
LOAD_MODEL = False
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.9

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
class ImageDataset(Dataset):
    """
    All image files found under `folder` (searched recursively, in sorted order),
    resized to `image_size` - the PyTorch counterpart of Keras'
    image_dataset_from_directory. Returns uint8 tensors of shape (C, H, W).
    """

    def __init__(self, folder, image_size, color_mode="rgb", labels=None):
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
        self.paths = sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in exts)
        self.mode = {"grayscale": "L", "rgb": "RGB"}[color_mode]
        self.resize = transforms.Resize(
            image_size, interpolation=transforms.InterpolationMode.BILINEAR
        )
        self.labels = labels
        self.transform = None  # optional function applied to each image
        self.target_transform = None  # optional function applied to each label
        print(f"Found {len(self.paths)} files.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert(self.mode)
        img = transforms.functional.pil_to_tensor(self.resize(img))
        if self.transform:
            img = self.transform(img)
        if self.labels is None:
            return img
        label = self.labels[idx]
        if self.target_transform:
            label = self.target_transform(label)
        return img, label


# Load the data
train_data = ImageDataset(
    "../../../data/celeba-dataset/img_align_celeba/img_align_celeba",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    color_mode="rgb",
)

In [ ]:
# Preprocess the data
def preprocess(img):
    """
    Normalize the images to [-1, 1]
    """
    img = (img.float() - 127.5) / 127.5
    return img


# Like Keras' .map(): preprocess is applied to every image as it is loaded.
# The DataLoader groups the images into shuffled batches.
train_data.transform = preprocess
train = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# Show some faces from the training set
train_sample = sample_batch(train)

In [ ]:
display(train_sample, cmap=None)

## 2. Build the WGAN-GP <a name="build"></a>

In [ ]:
# PyTorch images are channels-first: (batch, channels, height, width).
# padding=1 with kernel 4 / stride 2 halves the size, like Keras' padding="same".
critic = nn.Sequential(
    nn.Conv2d(CHANNELS, 64, kernel_size=4, stride=2, padding=1),
    nn.LeakyReLU(0.2),
    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
    nn.LeakyReLU(0.3),  # Keras' LeakyReLU() default slope is 0.3 (PyTorch's is 0.01)
    nn.Dropout(0.3),
    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0),
    nn.Flatten(),
).to(device)

print(critic)
print(f"Total params: {sum(p.numel() for p in critic.parameters()):,}")

In [ ]:
generator = nn.Sequential(
    nn.Unflatten(1, (Z_DIM, 1, 1)),  # (batch, Z_DIM) -> (batch, Z_DIM, 1, 1)
    nn.ConvTranspose2d(Z_DIM, 512, kernel_size=4, stride=1, padding=0, bias=False),
    # PyTorch momentum = 1 - Keras momentum (Keras 0.9 -> PyTorch 0.1)
    nn.BatchNorm2d(512, momentum=0.1),
    nn.LeakyReLU(0.2),
    # padding=1 with kernel 4 / stride 2 doubles the size, like Keras' padding="same"
    nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(128, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(64, CHANNELS, kernel_size=4, stride=2, padding=1),
    nn.Tanh(),
).to(device)

print(generator)
print(f"Total params: {sum(p.numel() for p in generator.parameters()):,}")

In [ ]:
class WGANGP(nn.Module):
    def __init__(self, critic, generator, latent_dim, critic_steps, gp_weight):
        super().__init__()
        self.critic = critic
        self.generator = generator
        self.latent_dim = latent_dim
        self.critic_steps = critic_steps
        self.gp_weight = gp_weight

    def gradient_penalty(self, real_images, fake_images):
        batch_size = real_images.shape[0]
        # (the book samples alpha from a normal distribution;
        # the WGAN-GP paper uses a uniform one on [0, 1])
        alpha = torch.randn(batch_size, 1, 1, 1, device=real_images.device)
        diff = fake_images - real_images
        interpolated = (real_images + alpha * diff).requires_grad_(True)

        pred = self.critic(interpolated)

        # Gradient of the critic's output w.r.t. its input images.
        # create_graph=True lets us backpropagate through the penalty itself.
        grads = torch.autograd.grad(pred.sum(), interpolated, create_graph=True)[0]
        norm = torch.sqrt(torch.sum(grads**2, dim=[1, 2, 3]))
        gp = torch.mean((norm - 1.0) ** 2)
        return gp

    def train_step(self, real_images, c_optimizer, g_optimizer):
        batch_size = real_images.shape[0]

        # Train the critic several times per generator update
        for i in range(self.critic_steps):
            random_latent_vectors = torch.randn(batch_size, self.latent_dim, device=real_images.device)
            with torch.no_grad():  # the generator is not trained in this step
                fake_images = self.generator(random_latent_vectors)

            fake_predictions = self.critic(fake_images)
            real_predictions = self.critic(real_images)

            c_wass_loss = fake_predictions.mean() - real_predictions.mean()
            c_gp = self.gradient_penalty(real_images, fake_images)
            c_loss = c_wass_loss + c_gp * self.gp_weight

            c_optimizer.zero_grad()  # reset gradients from the previous step
            c_loss.backward()  # backpropagation
            c_optimizer.step()  # update the critic weights

        # Train the generator
        random_latent_vectors = torch.randn(batch_size, self.latent_dim, device=real_images.device)
        fake_images = self.generator(random_latent_vectors)
        fake_predictions = self.critic(fake_images)
        g_loss = -fake_predictions.mean()

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

        return {
            "c_loss": c_loss.item(),
            "c_wass_loss": c_wass_loss.item(),
            "c_gp": c_gp.item(),
            "g_loss": g_loss.item(),
        }

In [ ]:
# Create a GAN
wgangp = WGANGP(
    critic=critic,
    generator=generator,
    latent_dim=Z_DIM,
    critic_steps=CRITIC_STEPS,
    gp_weight=GP_WEIGHT,
)

In [ ]:
if LOAD_MODEL:
    wgangp.load_state_dict(torch.load("./checkpoint/checkpoint.pt", map_location=device))

## 3. Train the GAN <a name="train"></a>

In [ ]:
# One optimizer per network
c_optimizer = torch.optim.Adam(
    critic.parameters(), lr=LEARNING_RATE, betas=(ADAM_BETA_1, ADAM_BETA_2)
)
g_optimizer = torch.optim.Adam(
    generator.parameters(), lr=LEARNING_RATE, betas=(ADAM_BETA_1, ADAM_BETA_2)
)

In [ ]:
# TensorBoard logs (like callbacks.TensorBoard); the checkpoint is saved in the training loop
writer = SummaryWriter(log_dir="./logs")


def generate_images(epoch, num_img=10, latent_dim=Z_DIM):
    """Saves and shows images from the generator (like the Keras ImageGenerator callback)"""
    generator.eval()  # BatchNorm uses its running statistics, like Keras' training=False
    with torch.no_grad():
        random_latent_vectors = torch.randn(num_img, latent_dim, device=device)
        generated_images = generator(random_latent_vectors)
    generator.train()
    generated_images = generated_images * 127.5 + 127.5
    display(
        generated_images,
        save_to="./output/generated_img_%03d.png" % (epoch),
        cmap=None,
    )

In [ ]:
steps_per_epoch = 2  # like Keras' steps_per_epoch: an "epoch" is just 2 batches

for epoch in range(EPOCHS):
    wgangp.train()
    totals = {}
    for real_images in itertools.islice(train, steps_per_epoch):
        logs = wgangp.train_step(real_images.to(device), c_optimizer, g_optimizer)
        for name, value in logs.items():
            totals[name] = totals.get(name, 0.0) + value

    # Average the metrics over the epoch, like Keras' progress bar
    logs = {name: total / steps_per_epoch for name, total in totals.items()}
    print(f"Epoch {epoch + 1}/{EPOCHS} - " + " - ".join(f"{k}: {v:.4f}" for k, v in logs.items()))

    # The work the Keras callbacks did at the end of each epoch
    for name, value in logs.items():
        writer.add_scalar(name, value, epoch)
    torch.save(wgangp.state_dict(), "./checkpoint/checkpoint.pt")
    generate_images(epoch)

writer.close()

In [ ]:
# Save the final models
torch.save(generator.state_dict(), "./models/generator.pt")
torch.save(critic.state_dict(), "./models/critic.pt")

## Generate images

In [ ]:
z_sample = np.random.normal(size=(10, Z_DIM))
wgangp.generator.eval()  # BatchNorm in inference mode, like Keras' predict()
with torch.no_grad():
    imgs = wgangp.generator(torch.tensor(z_sample, dtype=torch.float32, device=device))
display(imgs, cmap=None)